# Projet 4 — Analyse des performances et de la valeur marchande des joueurs de football

## A. Collecte des données (Web Scraping)

Cette section couvre l'intégralité de la collecte de données depuis **Transfermarkt.com** :
- Bibliothèques utilisées
- Fonction de scraping avec gestion des erreurs
- Boucle de collecte sur plusieurs classements (postes)
- Sauvegarde progressive des données


### Source des données

**Site source :** [Transfermarkt.com](https://www.transfermarkt.com)

**Page utilisée :** Classement "Most Valuable Players" (`marktwertetop`), filtré par poste
(`spielerposition_id`) pour obtenir un maximum de joueurs uniques, la version globale du
classement étant plafonnée à 500 joueurs.

**Méthode :** Scraping HTML avec `requests` + `BeautifulSoup`. Le tableau des joueurs
est directement présent dans le HTML retourné par le serveur (pas de JavaScript
dynamique sur cette page), ce qui permet un scraping simple et fiable.

**Variables collectées :**

| Variable | Description |
|---|---|
| `nom` | Nom du joueur |
| `poste` | Poste principal (Gardien, Défenseur central, Ailier droit...) |
| `age` | Âge du joueur |
| `nationalite` | Nationalité |
| `club` | Club actuel |
| `valeur_marchande` | Valeur marchande estimée (format texte, ex: `€90.00m`) |
| `matches_played` | Nombre de matchs joués (saison 2025/26, club uniquement) |
| `goals` | Buts marqués |
| `own_goals` | Buts contre son camp |
| `assists` | Passes décisives |
| `yellow_cards` | Cartons jaunes |
| `red_cards` | Cartons rouges |


### Bibliothèques utilisées

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# User-Agent pour simuler un navigateur réel (évite certains blocages basiques)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

### Fonction principale de scraping

Cette fonction scrape **une page** du classement (25 joueurs par page) et gère :
- **Les erreurs réseau et serveur** (`502`, `503`, `429`) avec un système de nouvelle
  tentative à délai progressif (*backoff*), car Transfermarkt bloque temporairement
  les requêtes trop fréquentes.
- **Les valeurs manquantes** : chaque champ est extrait avec une vérification
  (`if ... else None`), pour éviter que le script ne plante si une donnée est absente
  sur une ligne particulière.


In [4]:
def scrape_page_classement(url, tentatives=5):
    """
    Scrape une page du classement Transfermarkt (25 joueurs).

    Gestion des erreurs :
    - Retry automatique (jusqu'à `tentatives` fois) sur les erreurs serveur
      temporaires (502, 503, 429), avec un délai croissant entre chaque tentative.
    - Retourne une liste vide en cas d'échec définitif, plutôt que de lever une
      exception qui interromprait toute la collecte.
    """
    response = None
    for essai in range(tentatives):
        try:
            response = requests.get(url, headers=headers, timeout=15)
        except Exception as e:
            print(f"Erreur réseau : {e}")
            time.sleep(5)
            continue

        if response.status_code == 200:
            break
        elif response.status_code in (502, 503, 429):
            attente = 5 * (essai + 1)  # 5s, 10s, 15s, 20s, 25s...
            print(f"Erreur {response.status_code}, attente {attente}s avant nouvelle tentative...")
            time.sleep(attente)
            continue
        else:
            print(f"Erreur {response.status_code}")
            return []

    if response is None or response.status_code != 200:
        print("Échec définitif après plusieurs tentatives")
        return []

    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table", class_="items")
    if table is None:
        print("Tableau non trouvé sur cette page")
        return []

    rows = table.find_all("tr", class_=["odd", "even"])
    joueurs = []

    for row in rows:
        try:
            tds = row.find_all("td")
            if len(tds) < 18:
                continue  # ligne incomplète, on l'ignore proprement

            lien_tag = tds[3].find("a", href=True)
            nom = lien_tag.get_text(strip=True) if lien_tag else tds[3].get_text(strip=True)
            lien_profil = "https://www.transfermarkt.com" + lien_tag["href"] if lien_tag else None

            poste = tds[4].get_text(strip=True)
            age = tds[5].get_text(strip=True)

            nat_img = tds[6].find("img")
            nationalite = nat_img["title"] if nat_img and nat_img.has_attr("title") else None

            club_img = tds[7].find("img")
            club = club_img["title"] if club_img and club_img.has_attr("title") else None

            joueurs.append({
                "nom": nom,
                "poste": poste,
                "age": age,
                "nationalite": nationalite,
                "club": club,
                "valeur_marchande": tds[8].get_text(strip=True),
                "matches_played": tds[9].get_text(strip=True),
                "goals": tds[10].get_text(strip=True),
                "own_goals": tds[11].get_text(strip=True),
                "assists": tds[12].get_text(strip=True),
                "yellow_cards": tds[13].get_text(strip=True),
                "red_cards": tds[15].get_text(strip=True),
                "lien_profil": lien_profil
            })
        except Exception as e:
            print(f"Erreur sur une ligne : {e}")
            continue  # on passe à la ligne suivante sans interrompre le scraping

    return joueurs

### Construction des URLs

Chaque poste a un identifiant (`spielerposition_id`) propre côté Transfermarkt.
Utiliser un classement séparé par poste permet de dépasser la limite de 500 joueurs
du classement global.

In [5]:
def build_page_url(page_num, position_id):
    """Construit l'URL d'une page donnée pour un poste donné."""
    base = ("https://www.transfermarkt.com/spieler-statistik/wertvollstespieler/marktwertetop"
            f"?land_id=0&ausrichtung=alle&spielerposition_id={position_id}&altersklasse=alle"
            "&jahrgang=0&kontinent_id=0&jahr=2025&plus=1")
    if page_num == 1:
        return base
    return f"{base}&page={page_num}"


# IDs de poste vérifiés empiriquement (testés un par un avant utilisation)
POSTES = [
    ("Goalkeeper",          1),
    ("Centre-Back",         3),
    ("Left-Back",           4),
    ("Right-Back",          5),
    ("Defensive Midfield",  6),
    ("Central Midfield",    7),
    ("Attacking Midfield", 10),
    ("Left Winger",        11),
    ("Right Winger",       12),
    ("Centre-Forward",     14),
]

### Boucle de collecte et sauvegarde

La collecte se fait poste par poste, page par page (jusqu'à 20 pages = 500 joueurs
par poste). Le fichier CSV est **sauvegardé après chaque page** : en cas de blocage
temporaire du site en cours de route, aucune donnée déjà collectée n'est perdue.

In [6]:
FICHIER_SORTIE = "dataset.csv"
PAGE_FIN = 20  # 20 pages x 25 joueurs = jusqu'à 500 joueurs par poste

# Reprise : on charge les données déjà collectées si le fichier existe déjà
try:
    df_existant = pd.read_csv(FICHIER_SORTIE, sep=";", encoding="utf-8-sig")
    toutes_les_donnees = df_existant.to_dict("records")
    print(f"Reprise : {len(toutes_les_donnees)} joueurs déjà présents")
except FileNotFoundError:
    toutes_les_donnees = []
    print("Aucun fichier existant, démarrage de la collecte")

for nom_poste, position_id in POSTES:
    print(f"\n--- Poste : {nom_poste} ---")

    for page_num in range(1, PAGE_FIN + 1):
        url = build_page_url(page_num, position_id)
        joueurs = scrape_page_classement(url)

        if len(joueurs) == 0:
            print(f"Fin des pages pour {nom_poste} (page {page_num})")
            break

        toutes_les_donnees.extend(joueurs)

        # Déduplication + sauvegarde après chaque page
        df_temp = pd.DataFrame(toutes_les_donnees).drop_duplicates(subset="lien_profil")
        df_temp.to_csv(FICHIER_SORTIE, index=False, sep=";", encoding="utf-8-sig")

        time.sleep(3)  # pause entre chaque requête, pour ne pas surcharger le serveur

print(f"\n=== Collecte terminée : {df_temp.shape[0]} joueurs uniques ===")

Aucun fichier existant, démarrage de la collecte

--- Poste : Goalkeeper ---

--- Poste : Centre-Back ---

--- Poste : Left-Back ---

--- Poste : Right-Back ---

--- Poste : Defensive Midfield ---

--- Poste : Central Midfield ---

--- Poste : Attacking Midfield ---

--- Poste : Left Winger ---

--- Poste : Right Winger ---

--- Poste : Centre-Forward ---
Erreur réseau : HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)

=== Collecte terminée : 4621 joueurs uniques ===


### Vérification finale du dataset collecté

In [9]:
df_final = pd.read_csv("dataset1.csv", sep=",", encoding="utf-8-sig")

print(f"Nombre total de joueurs collectés : {df_final.shape[0]}")
print(f"Colonnes : {list(df_final.columns)}")
print(f"\nValeurs manquantes par colonne :")
print(df_final.isnull().sum())

df_final.head(10)

Nombre total de joueurs collectés : 2159
Colonnes : ['nom', 'poste', 'age', 'nationalite', 'club', 'valeur_marchande', 'matches_played', 'goals', 'own_goals', 'assists', 'yellow_cards', 'red_cards']

Valeurs manquantes par colonne :
nom                 0
poste               0
age                 0
nationalite         0
club                0
valeur_marchande    0
matches_played      0
goals               0
own_goals           0
assists             0
yellow_cards        0
red_cards           0
dtype: int64


,nom,poste,age,nationalite,club,valeur_marchande,matches_played,goals,own_goals,assists,yellow_cards,red_cards
0,Lamine Yamal,Right Winger,17,Spain,FC Barcelona,€200.00m,54,21,0,24,5,0
1,Bukayo Saka,Right Winger,23,England,Arsenal FC,€150.00m,38,10,0,5,3,0
2,Michael Olise,Right Winger,23,France,Bayern Munich,€130.00m,57,21,0,28,7,0
3,Rodrygo,Right Winger,23,Brazil,Real Madrid,€90.00m,54,10,0,11,4,0
4,Désiré Doué,Right Winger,19,France,Paris Saint-Germain,€90.00m,54,19,0,16,3,0
5,Estêvão,Right Winger,17,Brazil,Sociedade Esportiva Palmeiras,€80.00m,60,17,0,6,11,0
6,Bryan Mbeumo,Right Winger,25,Cameroon,Brentford FC,€75.00m,38,17,0,7,4,0
7,Antoine Semenyo,Right Winger,24,Ghana,AFC Bournemouth,€65.00m,42,17,0,8,9,0
8,Pedro Neto,Right Winger,24,Portugal,Chelsea FC,€60.00m,56,12,0,7,7,0
9,Karim Adeyemi,Right Winger,22,Germany,Borussia Dortmund,€60.00m,53,13,0,9,9,0
